# Mini-Project 3 — A Framework RAG with LangChain
### IT7075: Applied AI for Cybersecurity · LangChain

In Mini-Project 2 you built RAG **by hand**: your own chunker, your own Chroma calls, your own
prompt assembly, your own retrieve-then-generate function. Now you build the same thing **with
LangChain**, and the point of the exercise is to see exactly which parts the framework takes over.

Five pieces are left for you to write, one per stage of the pipeline in the langchain_rag notebook:

| | Stage | You write |
|---|---|---|
| **TODO 1** | load the documents and split them | 3 lines |
| **TODO 2** | build the Chroma vector store | 1 line |
| **TODO 3** | turn it into a retriever (`k` + score threshold) | 2 lines |
| **TODO 4** | write the prompt template | 2 lines |
| **TODO 5** | wire the LCEL chain together with `\|` | 3 lines |

Find them by searching for `###################`:

```
###################  TODO n — title  ###################
# what to do, and how
################
```

> **Runs offline.** Embeddings are the local `all-MiniLM-L6-v2` model, so retrieval never needs an
> API key. If you have a working `OPENAI_API_KEY` the last stage of the chain calls a real model;
> if you don't — or it is out of credit — the notebook substitutes an offline answer and keeps
> going. Everything you are graded on works either way.

## Part 1 — Load and split the documents  ·  **TODO 1**

LangChain's loaders return **`Document` objects** (text plus metadata), not plain strings — that is
the first difference from Mini-Project 2. The splitter then works on documents and hands back more
documents, carrying the metadata along for you.

> **Expected output:** `1 document -> 10 chunks`.

In [ ]:
# If needed: !pip install -q langchain langchain-community langchain-chroma langchain-huggingface langchain-openai
import json, os, warnings
import pandas as pd

# Keep this notebook on the CPU. The models here are tiny (milliseconds either way),
# and this avoids the "CUDA error: no kernel image is available" failure -- or a very
# long hang while torch initialises -- on a machine whose PyTorch build does not match
# its GPU. This must run BEFORE anything imports torch.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "")

# Two noisy-but-harmless warnings, silenced so the output stays readable:
#  * langchain-community is being retired (TextLoader still works and is what the langchain_rag notebook uses)
#  * LangChain warns when a relevance score falls outside 0-1. With these embeddings an
#    UNRELATED question scores slightly below 0 -- which is the behaviour we want. See Part 3.
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", message="Relevance scores must be between")

# LangChain also logs "No relevant docs were retrieved..." every time the threshold
# rejects everything. That is the behaviour we are measuring, and the cells below
# already report it as "retrieved 0 chunk(s)", so we log it once instead of 20 times.
import logging
logging.getLogger("langchain_core.vectorstores.base").setLevel(logging.ERROR)
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter

# ---- the knowledge base and question set that came with the course -------
def course_path(name, folders=("../../code/", "../../../code/", "./")):
    for f in folders:
        if os.path.exists(f + name):
            return f + name
    return name

KB_FILE = course_path("ai_framework_kb.md")
GOLDSET_FILE = ("framework_goldset.json" if os.path.exists("framework_goldset.json")
                else "../framework_goldset.json")


###################  TODO 1 — load the file and split it into chunks  ###################
# WHAT: load KB_FILE into `documents`, then split it into `chunks`.
#
# HOW (the "Load & Split" cell in the langchain_rag notebook):
#   * loader    = TextLoader(KB_FILE, encoding="utf-8")
#   * documents = loader.load()
#         -> a LIST of Document objects. A whole .md file loads as ONE document,
#            so len(documents) is 1. The text is in documents[0].page_content.
#   * splitter  = CharacterTextSplitter(chunk_size=900, chunk_overlap=150,
#                                       separator="\n")      <- given below
#   * chunks    = splitter.split_documents(documents)
#         -> note split_documentS: it takes Documents and returns Documents,
#            keeping the metadata. (split_text() would take a plain string.)
#
# WHY separator="\n": the splitter tries to break on that separator first, so
# chunks end at line boundaries instead of mid-sentence like in Mini-Project 2.
################

loader = ____                                                                   # <<< 1 line
documents = ____                                                                # <<< 1 line
splitter = CharacterTextSplitter(chunk_size=900, chunk_overlap=150, separator="\n")
chunks = ____                                                                   # <<< 1 line

print(len(documents), "document ->", len(chunks), "chunks")
print("\nFirst chunk:\n", chunks[0].page_content[:200], "...")
print("\nIts metadata:", chunks[0].metadata)

## Part 2 — Build the vector store  ·  **TODO 2**

In Mini-Project 2 this took three steps: embed the chunks, create a collection, then add ids,
documents, embeddings and metadata. LangChain does all of it in **one call** — `Chroma.from_documents`
embeds and stores in a single line. That is the framework's whole pitch.

We use the **local** embedding model, so no API key is needed.

> **Expected output:** `stored 10 chunks`.

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# The offline embedding model the langchain_rag notebook names in its comment.
# Swap in OpenAIEmbeddings(model="text-embedding-3-small") if you have a working key.
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


###################  TODO 2 — put the chunks into Chroma  ###################
# WHAT: build the vector store in ONE line.
#
# HOW (the "Embed & Store" cell in the langchain_rag notebook):
#       Chroma.from_documents(<the chunks>, <the embeddings>)
#
# That single call embeds every chunk with `embeddings` and stores the vectors,
# the text, and the metadata. Compare it with Mini-Project 2, where you called
# embed() yourself and then collection.add(ids=..., documents=..., embeddings=...,
# metadatas=...). Same work; the framework hides it.
#
# NOTE: we leave out persist_directory on purpose, so the store lives in memory and
#       re-running this cell can never mix new chunks with stale ones.
################

db = ____                                                          # <<< 1 line

print("stored", db._collection.count(), "chunks")

## Part 3 — Make a retriever  ·  **TODO 3**

A **retriever** is a vector store wrapped in a standard interface, so anything downstream can just
call `.invoke(question)`. It is also where your two Mini-Project 2 knobs reappear under new names:

| Mini-Project 2 (by hand) | LangChain |
|---|---|
| `k` = how many chunks | `search_kwargs={"k": ...}` |
| `max_distance` — keep hits **below** it | `score_threshold` — keep hits **above** it |

Note the flip: a **distance** is small when things are similar, but LangChain's **relevance score**
is large when things are similar. Same idea, opposite direction.

> **Expected output:** 3 chunks for the NIST question, **0** for "the capital of France" — the
> refusal you built by hand in Mini-Project 2, now handled by the retriever.

In [ ]:
###################  TODO 3 — wrap the store in a retriever  ###################
# WHAT: create `retriever` from `db`, keeping at most K chunks and only those
#       scoring above THRESHOLD.
#
# HOW (the "Make a Retriever" cell in the langchain_rag notebook):
#       db.as_retriever(
#           search_type="similarity_score_threshold",
#           search_kwargs={"k": K, "score_threshold": THRESHOLD})
#
#   * search_type is the STRING "similarity_score_threshold"
#   * search_kwargs is a DICT with two keys: "k" and "score_threshold"
#
#   * search_type="similarity_score_threshold" is what makes score_threshold apply.
#     With the default search_type="similarity" the threshold is IGNORED and you
#     always get K chunks back -- the bug that made your Mini-Project 2 bot answer
#     questions its documents could not support.
#   * relevance score: HIGHER = more similar (the opposite of a distance).
#   * use the constants K and THRESHOLD below, not literal numbers -- Part 6
#     changes them.
################

K = 3
THRESHOLD = 0.3          # the langchain_rag value. Part 6 asks whether it is right.

retriever = db.as_retriever(
    search_type=____,                                              # <<< 1 line
    search_kwargs=____)                                            # <<< 1 line


# ---- given: see what comes back, and what does not ----------------------
for q in ["What are the four core functions of the NIST AI Risk Management Framework?",
          "What is the capital of France?"]:
    got = retriever.invoke(q)
    print("Q:", q)
    print("   retrieved", len(got), "chunk(s)")
    for d in got:
        print("     -", d.page_content[:70].strip().replace("\n", " "), "...")
    print()

## Part 4 — The prompt template  ·  **TODO 4**

A `ChatPromptTemplate` is the reusable prompt from the langchain_basics notebook: you write it once with `{}`
placeholders, and the chain fills them in on every question. This replaces the string `.format()`
you did by hand in Mini-Project 2.

> **Expected output:** the filled-in prompt, showing your context in the system message and the
> question in the human message.

In [ ]:
###################  TODO 4 — write the prompt template  ###################
# WHAT: build `rag_prompt` with a system message and a human message.
#
# HOW (prompt templates in langchain_basics + the chain cell in langchain_rag):
#       rag_prompt = ChatPromptTemplate.from_messages([
#           ("system", "....{context}...."),
#           ("human", "{question}"),
#       ])
#
# The system message must:
#   * tell the model to answer using ONLY the context
#   * contain the placeholder {context}
#   * say to start the reply with [RAG] when the answer is in the context, and with
#     [LLM] plus "I don't know" when it is not -- the same tagging as Mini-Project 2
# The human message is just the placeholder {question}.
#
# WATCH OUT: the two placeholder names must be exactly `context` and `question`,
#            because that is what the chain in Part 5 feeds in.
################

from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", ____),                                                            # <<< 1 line
    ("human", ____),                                                             # <<< 1 line
])

# ---- given: check what the filled-in prompt looks like ------------------
preview = rag_prompt.invoke({"context": "(pretend context here)", "question": "test question"})
for m in preview.to_messages():
    print("[%s] %s\n" % (m.type, m.content[:220]))

## Part 5 — Wire the chain with `|`  ·  **TODO 5**

This is the part that has no equivalent in Mini-Project 2. There you wrote an `ask()` function that
called retrieve, then built the context, then called generate. In LangChain you **declare** that
flow with the pipe operator and get a single runnable object back:

```
{"context": retriever | format_docs, "question": RunnablePassthrough()}  |  prompt  |  model  |  parser
```

Read it left to right: the dict runs the retriever **and** passes the raw question through, the
prompt fills both placeholders, the model answers, and the parser turns the reply into a string.

> **Expected output:** the NIST question answered and tagged `[RAG]`; "the capital of France"
> refused and tagged `[LLM]`, because the retriever returned nothing for it.

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda


def format_docs(docs):
    """GIVEN: turn the retrieved Documents into one block of text for {context}."""
    return "\n\n".join(d.page_content for d in docs)


def offline_answer(prompt_value):
    """GIVEN: stands in for the model when no LLM is available (a RunnableLambda,
    exactly like the ones in langchain_basics). It reads the context out of the filled-in
    prompt and quotes it, so the chain still produces a tagged answer."""
    context = prompt_value.to_messages()[0].content.split("# Context", 1)[-1].strip()
    if context == "":
        return "[LLM] I don't know -- the knowledge base does not cover this. (offline mode)"
    return "[RAG] (offline mode) From the retrieved context: " + context[:220]


def build_model():
    """GIVEN: a real chat model if a working key is available, else the offline stand-in."""
    if os.getenv("OPENAI_API_KEY"):
        try:
            from langchain_openai import ChatOpenAI
            m = ChatOpenAI(model="gpt-4o-mini", temperature=0)
            m.invoke("ping")                       # fail fast if the key has no credit
            print("using ChatOpenAI(gpt-4o-mini)")
            return m | StrOutputParser()
        except Exception as e:
            print("(LLM unavailable - %s. Using offline mode.)" % type(e).__name__)
    else:
        print("(no OPENAI_API_KEY - using offline mode.)")
    return RunnableLambda(offline_answer)


model = build_model()


###################  TODO 5 — build the LCEL chain  ###################
# WHAT: assemble `rag_chain` from the four pieces you already have, using `|`.
#
# HOW (the "Build the RAG Chain (LCEL)" cell in the langchain_rag notebook):
#       rag_chain = (
#           {"context": <retriever piped into format_docs>,
#            "question": <pass the question through untouched>}
#           | <your prompt>
#           | <the model>
#       )
#
#   * the DICT is the interesting bit: both entries receive the question you invoke
#     the chain with. "context" sends it through the retriever and then through
#     format_docs; "question" passes it along untouched (RunnablePassthrough()).
#     The result is {"context": "...", "question": "..."} -- exactly the two
#     placeholders your prompt needs.
#   * `model` here already includes the output parser, so you do not add one.
#
# THEN: invoke it with just a question string -- rag_chain.invoke("...?")
################

rag_chain = (
    ____                                                                         # <<< 1 line
    | ____                                                                       # <<< 1 line
    | ____                                                                       # <<< 1 line
)

# ---- given: one question the KB answers, one it does not ----------------
for q in ["What are the four core functions of the NIST AI Risk Management Framework?",
          "What is the capital of France?"]:
    print("Q:", q)
    print("A:", rag_chain.invoke(q)[:300], "\n")

## Part 6 — Ask the bot, and measure it  *(no TODO — run it and read it)*

`framework_goldset.json` holds **8 questions** about the provided knowledge base: **6 it answers**
(each with a phrase that proves it) and **2 it does not**. The two counts are the same ones you
used in Mini-Project 2:

- **found** — of the 6, how many retrieved a chunk containing the evidence?
- **refused** — of the 2, how many correctly retrieved *nothing*?

> **Expected output at the langchain_rag settings (`k=3`, threshold 0.3): found 5/6, refused 1/2.**
> Two things go wrong at once, and neither is a bug in your code:
> **Q2** (MITRE ATLAS) finds nothing, and **Q8** — which the knowledge base cannot answer —
> gets context anyway. Part 7 shows why, and why you cannot fix both by tuning.

In [ ]:
GOLDSET = json.load(open(GOLDSET_FILE))["questions"]
N_ANS = sum(q["answerable"] for q in GOLDSET)
N_UNANS = len(GOLDSET) - N_ANS


def score(retriever, goldset):
    """GIVEN: count found / refused, the same two numbers as Mini-Project 2."""
    found, refused, detail = 0, 0, []
    for q in goldset:
        docs = retriever.invoke(q["question"])
        text = " ".join(d.page_content for d in docs).lower()
        hit = any(p.lower() in text for p in q["evidence"])
        if q["answerable"]:
            found += 1 if hit else 0
        else:
            refused += 1 if len(docs) == 0 else 0
        detail.append({"qid": q["qid"], "answerable": q["answerable"],
                       "chunks": len(docs), "evidence_found": hit if q["answerable"] else "-"})
    return found, refused, detail


found, refused, detail = score(retriever, GOLDSET)
print("found   %d/%d" % (found, N_ANS))
print("refused %d/%d\n" % (refused, N_UNANS))
print(pd.DataFrame(detail).to_string(index=False))

## Part 7 — Tune the threshold  *(no TODO)*

Sweep the threshold and watch the two counts fight each other.

> **Expected output:**
>
> | threshold | found | refused | what changed |
> |---|---:|---:|---|
> | 0.1 | **6/6** | 1/2 | finds everything, but Q8 still gets context |
> | 0.2 | **6/6** | 1/2 | same |
> | **0.3** *(lecture default)* | 5/6 | 1/2 | **worst of both** — loses Q2 *and* still leaks Q8 |
> | 0.4 | 5/6 | 1/2 | |
> | 0.5 | 5/6 | 1/2 | |
> | 0.6 | 4/6 | **2/2** | finally refuses Q8 — by giving up Q2 and Q3 too |
>
> **Read that table carefully: there is no good row.** No threshold gets 6/6 *and* 2/2, and the
> last cell shows you why — Q8, which your documents **cannot** answer, scores **0.505**, while
> Q2, which they answer perfectly well, scores only **0.258**. The wrong answer looks *more*
> relevant than the right one, so no single cut-off can separate them.
>
> That is the honest finding of this project, and it is worth more than a tidy result: **tuning
> cannot fix a ranking that is already wrong.** Fixing it would need better chunking, a better
> embedding model, or a reranking step — not a different number.
>
> Mini-Project 2 showed you the other half of this trade-off: there a *loose* threshold let the
> bot answer what it shouldn't. Both failures are silent, and only a question set catches either.

In [ ]:
rows = []
for t in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6]:
    r = db.as_retriever(search_type="similarity_score_threshold",
                        search_kwargs={"k": K, "score_threshold": t})
    f, ref, _ = score(r, GOLDSET)
    rows.append({"threshold": t, "found": "%d/%d" % (f, N_ANS),
                 "refused": "%d/%d" % (ref, N_UNANS)})

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("results.csv", index=False)
print("\nwrote results.csv")

# Why is there no good row? Compare the best score for a question the KB ANSWERS
# with the best score for one it CANNOT answer.
print("\ntop-1 relevance score per question (higher = looks more relevant):")
for q in GOLDSET:
    s = db.similarity_search_with_relevance_scores(q["question"], k=1)[0][1]
    print("   %-4s %-7s %+.3f  %s" % (q["qid"], "answer" if q["answerable"] else "ABSENT",
                                      s, q["question"][:52]))
print("\nQ8 is not in the knowledge base yet outranks Q2, which is. No cut-off fixes that.")

## Part 8: rebuild your own RAG with LangChain

Everything above used the provided knowledge base, and it was practice. Now build the same system you
built in the RAG module, this time with LangChain, using the same data and the same questions.

Copy your `my_kb/` folder and your `goldset.json` from the RAG module into this project folder. Do not
change the documents or the questions: the comparison in your report depends on both systems answering
exactly the same eight questions over exactly the same text.

No code is provided from here on. Add cells below that:

1. Load your documents with a LangChain loader and split them with a text splitter.
2. Build a vector store from the chunks and wrap it in a retriever.
3. Write a prompt template and wire the chain with the pipe operator.
4. Answer your eight questions and print the transcript, including one the documents cannot answer.
5. Score the same measurements you used in the RAG module, vary one setting at a time, and save the
   results as `results_langchain.csv`.

Note as you go which of your RAG-module functions each LangChain object replaced. That list is the
heart of your report.

In [ ]:
###################  YOUR RAG, REBUILT WITH LANGCHAIN  ###################
# Load your documents from my_kb/, split them, build a vector store and a
# retriever, write a prompt template, and wire the chain with the pipe.
################

### Answer your questions and measure the result

In [ ]:
###################  ANSWER AND MEASURE  ###################
# Answer your eight questions and print the transcript, including one the
# documents cannot answer. Then score the configurations you want to compare
# and save the table as results_langchain.csv.
################

---

## What to write up

One PDF report and a video of five to ten minutes. See `MiniProject_FrameworkRAG.md` for the full
requirements. The questions that matter:

- What did LangChain replace? Name the code from your RAG-module notebook that each LangChain object
  made unnecessary. Where is that a genuine win, and where does it hide something you would rather see?
- On the same documents and the same questions, how do the two systems compare on the numbers you
  measured, and how much work did each take to build?
- Which question is hardest for this version, and what does it retrieve instead?
- The `score_threshold` here is a relevance score, not the distance you used in the RAG module. Higher
  means more similar. Why does that flip the comparison, and what happens if you get it backwards?

Submit one PDF report with the link to your project folder and the link to your video on its first page.
The notebook, your documents, `goldset.json`, and your results live in the repository, which is private,
with your instructor and the TA added as collaborators. The notebook must be committed with its outputs
showing.